In [1]:
from pprint import pprint

from ollama import chat

from tools import Top_Type, code_extraction, linter, netlist




In [2]:
Ollama_MODEL="qwen2.5-coder:7b"

# Checking for the presence of mdoel and laoding the model 


In [3]:
stream=chat(model=Ollama_MODEL,
            messages=[{"role":"user","content":"What is your knowledge cut off and how  good are you with verilog and symbolic model verification coding ?"}],
            stream=True)
print("Printing the answer\n")
for chunk in stream:
    print(chunk['message']['content'],end='',flush=True)
print()

Printing the answer

My knowledge cutoff date is October 2023. I can provide information and answer questions based on data available up to that time.

Regarding Verilog and symbolic model verification, I have been trained on a variety of topics, including digital design concepts, hardware description languages (HDLs), and some aspects of formal verification techniques. However, my capabilities are based on the training data up to my cutoff date, which may not include the most recent developments or advanced features in Verilog or symbolic model verification tools.

If you have specific questions or need assistance with a particular aspect of Verilog or symbolic model verification, I can try to provide guidance based on what I've learned up until that point. Keep in mind that my responses are not guaranteed to be up-to-date or as comprehensive as those from someone who is actively working with these technologies.


In [4]:
_VERILOG_SYSTEM = """
ROLE:
Senior RTL designer; Verilog-2001 only.

INSTRUCTIONS:
Implement the user's specification exactly.
Generate clean, synthesizable RTL.
Use explicit signal widths and correct signedness.
Use always @(*) with '=' for combinational logic.
Use always @(posedge clk) with '<=' for sequential logic.
Assign all combinational outputs on every path.
Every case requires default.
Use $signed()/ $unsigned().

STEPS:
Analyze → generate → internally check syntax, widths,
signedness, latches, and synthesizability.

END:
Return ONLY complete Verilog-2001 code.
No markdown or explanation.

NEGATIVE:
No SystemVerilog, VHDL, logic, always_comb, always_ff,
typedef, enum, struct, interface, class, package,
assertions, dynamic arrays, declarations inside always blocks,
or expression/function-call slicing.
"""

In [5]:
def code_gen(prompt:str)->str:
    """ 
    INPUT:
    prompt: excepts user promtp a string providing the detials about the verilog code to be written by the llm.

    OUTPTS:
        gives the code as output 
    """
    stream=chat(model=Ollama_MODEL,messages=[{'role':'System',
                                           "content":f"""You are a Senoir Design Engineer and well versed with RTL code, 
                                           provide a clean,synthesizable,well commented,compact Verilog code with respect to the user request.
                                           DO not provide any explanation , provide only the verilog  code.
                                           Follow the rules{_VERILOG_SYSTEM}
                                           """},
                                           {"role":'user',
                                            "content":f"{prompt}"}],
                                            stream=True,
                                            options={
                                                    "temperature":0.5,
                                                     "seed":0
                                                     })
    resp=""
    for chunk in stream:
        content=chunk["message"]["content"]
        print(content,end="",flush=True)
        resp+=content
    return resp

In [6]:
User_inpt=input("Prompt the verilog module  requesting")
response=code_gen(User_inpt)

```verilog
module gcd_32bit (
    input wire clk,
    input wire rst_n,
    input wire start,
    input wire [31:0] a,
    input wire [31:0] b,
    output reg [31:0] result,
    output reg done
);

reg [31:0] dividend;
reg [31:0] divisor;
reg [31:0] temp;

always @(posedge clk or negedge rst_n) begin
    if (!rst_n) begin
        dividend <= 0;
        divisor <= 0;
        result <= 0;
        done <= 0;
    end else begin
        if (start) begin
            dividend <= a;
            divisor <= b;
            done <= 0;
        end else if (!done) begin
            if (divisor == 0) begin
                result <= dividend;
                done <= 1;
            end else begin
                temp <= $unsigned(dividend);
                dividend <= $unsigned(divisor);
                divisor <= temp % $unsigned(divisor);
            end
        end
    end
end

endmodule
```

In [7]:
clean_Vcode=code_extraction(response)
print(clean_Vcode)

module gcd_32bit (
    input wire clk,
    input wire rst_n,
    input wire start,
    input wire [31:0] a,
    input wire [31:0] b,
    output reg [31:0] result,
    output reg done
);

reg [31:0] dividend;
reg [31:0] divisor;
reg [31:0] temp;

always @(posedge clk or negedge rst_n) begin
    if (!rst_n) begin
        dividend <= 0;
        divisor <= 0;
        result <= 0;
        done <= 0;
    end else begin
        if (start) begin
            dividend <= a;
            divisor <= b;
            done <= 0;
        end else if (!done) begin
            if (divisor == 0) begin
                result <= dividend;
                done <= 1;
            end else begin
                temp <= $unsigned(dividend);
                dividend <= $unsigned(divisor);
                divisor <= temp % $unsigned(divisor);
            end
        end
    end
end

endmodule


In [8]:
res=linter(clean_Vcode)
print(res)

{'return_code': 0, 'stdout': '', 'stderr': ''}


In [9]:
top=Top_Type(clean_Vcode)

In [10]:
pprint(top)

{'err': '',
 'output': '\n'
           ' '
           '/----------------------------------------------------------------------------\\\n'
           ' |  yosys -- Yosys Open SYnthesis '
           'Suite                                       |\n'
           ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
           '<claire@yosyshq.com>         |\n'
           ' |  Distributed under an ISC-like license, type "license" to see '
           'terms        |\n'
           ' '
           '\\----------------------------------------------------------------------------/\n'
           ' Yosys 0.62+112 (git sha1 687a36af3, g++ 13.3.0-6ubuntu2~24.04.1 '
           '-fPIC -O3)\n'
           '\n'
           '-- Running command ` \n'
           '                             read_verilog /tmp/tmpe2imhgqe.v;\n'
           '                             proc;\n'
           '                             stat;\n'
           "                            ' --\n"
           '\n'
           '1. Executing V

In [11]:
top_name=top['top_module']
print(top_name)

gcd_32bit


In [12]:
net_out=netlist(top=top_name,module_path=f"./src/{top_name}.v",lib_path="../Nangate45/Nangate45_typ.lib")

In [13]:
pprint(net_out)

{'netlist_returncode': 0,
 'netlist_stderr': '',
 'netlist_stdout': '\n'
                   ' '
                   '/----------------------------------------------------------------------------\\\n'
                   ' |  yosys -- Yosys Open SYnthesis '
                   'Suite                                       |\n'
                   ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
                   '<claire@yosyshq.com>         |\n'
                   ' |  Distributed under an ISC-like license, type "license" '
                   'to see terms        |\n'
                   ' '
                   '\\----------------------------------------------------------------------------/\n'
                   ' Yosys 0.62+112 (git sha1 687a36af3, g++ '
                   '13.3.0-6ubuntu2~24.04.1 -fPIC -O3)\n'
                   '\n'
                   '-- Running command `\n'
                   '            read_verilog ./src/gcd_32bit.v;\n'
                   '            hierarchy -t